In [ ]:
import os
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx8G"

from datetime import datetime, timedelta
from pathlib import Path
import json

import geopandas as gpd
import pandas as pd
import r5py

In [ ]:
WGS84_CRS = "EPSG:4326"
RUN_DATE = datetime(2026, 3, 20)
DEPARTURE_TIMES = [9, 12, 17, 22]
MAX_TIME_MINUTES = 240

# Smaller chunks are safer for memory; larger chunks reduce r5py startup overhead.
ORIGIN_CHUNK_SIZE = 250

GRID_POINTS_FILE = Path("outputs/grid_points_3000m.parquet")
OSM_FILE = Path("data/switzerland-latest.osm.pbf")
GTFS_FILE = Path("data/gtfs/gtfs_filtered_no_taxi.zip")

OUTPUT_DIR = Path("outputs_ptA_ptB")
CHUNKS_DIR = OUTPUT_DIR / "chunks"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def time_label(hour: int) -> str:
    return f"{hour:02d}00"


def chunk_bounds(n_rows: int, chunk_size: int):
    for start in range(0, n_rows, chunk_size):
        end = min(start + chunk_size, n_rows)
        yield start, end

In [ ]:
required_files = [GRID_POINTS_FILE, OSM_FILE, GTFS_FILE]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

routing_points = gpd.read_parquet(GRID_POINTS_FILE)
routing_points["id"] = routing_points["id"].astype(str)

if routing_points.crs is None:
    routing_points = routing_points.set_crs(WGS84_CRS)
else:
    routing_points = routing_points.to_crs(WGS84_CRS)

routing_points = routing_points.sort_values("id").reset_index(drop=True)
routing_points = routing_points[["id", "geometry"]].copy()

print("Routing points:", len(routing_points))
print("Expected OD pairs per departure before r5 pruning:", len(routing_points) * len(routing_points))
print("Departure times:", [f"{hour:02d}:00" for hour in DEPARTURE_TIMES])

In [ ]:
network = r5py.TransportNetwork(
    str(OSM_FILE),
    [str(GTFS_FILE)],
)

In [ ]:
def chunk_file_for(departure_label: str, start: int, end: int) -> Path:
    return (
        CHUNKS_DIR
        / f"travel_times_ptA_ptB_3000m_departure_{departure_label}_max_240min_origins_{start:05d}_{end - 1:05d}.parquet"
    )


def compute_or_load_chunk(departure_hour: int, start: int, end: int) -> pd.DataFrame:
    departure_label = time_label(departure_hour)
    out_file = chunk_file_for(departure_label, start, end)

    if out_file.exists():
        print("Loading existing chunk:", out_file.name)
        return pd.read_parquet(out_file)

    origins_chunk = routing_points.iloc[start:end].copy()
    print(
        f"Computing {departure_label}, origins {start}-{end - 1} "
        f"({len(origins_chunk)} x {len(routing_points)} OD candidates)..."
    )

    matrix = r5py.TravelTimeMatrix(
        network,
        origins=origins_chunk,
        destinations=routing_points,
        departure=datetime(RUN_DATE.year, RUN_DATE.month, RUN_DATE.day, departure_hour, 0),
        departure_time_window=timedelta(minutes=60),
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        percentiles=[50],
        snap_to_network=True,
        max_time=timedelta(minutes=MAX_TIME_MINUTES),
    )

    matrix = matrix.copy()
    matrix["departure_label"] = departure_label
    matrix["departure_hour"] = departure_hour
    matrix["max_time_minutes"] = MAX_TIME_MINUTES
    matrix["origin_chunk_start"] = start
    matrix["origin_chunk_end"] = end - 1

    matrix.to_parquet(out_file, index=False)
    print("Saved:", out_file)
    return matrix

In [ ]:
manifest_records = []
origin_chunk_index_records = []

for departure_hour in DEPARTURE_TIMES:
    departure_label = time_label(departure_hour)

    for start, end in chunk_bounds(len(routing_points), ORIGIN_CHUNK_SIZE):
        chunk_path = chunk_file_for(departure_label, start, end)
        matrix = compute_or_load_chunk(departure_hour, start, end)

        manifest_records.append(
            {
                "departure_label": departure_label,
                "departure_hour": departure_hour,
                "max_time_minutes": MAX_TIME_MINUTES,
                "origin_chunk_start": start,
                "origin_chunk_end": end - 1,
                "file": str(chunk_path),
                "n_rows": int(len(matrix)),
                "n_non_null_travel_times": int(matrix["travel_time"].notna().sum()),
            }
        )

        for origin_id in routing_points.iloc[start:end]["id"]:
            origin_chunk_index_records.append(
                {
                    "departure_label": departure_label,
                    "departure_hour": departure_hour,
                    "from_id": str(origin_id),
                    "origin_chunk_start": start,
                    "origin_chunk_end": end - 1,
                    "file": str(chunk_path),
                }
            )

manifest = pd.DataFrame(manifest_records)
origin_chunk_index = pd.DataFrame(origin_chunk_index_records)

manifest.to_parquet(OUTPUT_DIR / "manifest.parquet", index=False)
origin_chunk_index.to_parquet(OUTPUT_DIR / "origin_chunk_index.parquet", index=False)

metadata = {
    "run_date": RUN_DATE.strftime("%Y-%m-%d"),
    "departure_times": [f"{hour:02d}:00" for hour in DEPARTURE_TIMES],
    "max_time_minutes": MAX_TIME_MINUTES,
    "origin_chunk_size": ORIGIN_CHUNK_SIZE,
    "n_points": int(len(routing_points)),
    "expected_od_pairs_per_departure_before_r5_pruning": int(len(routing_points) * len(routing_points)),
    "transport_modes": ["WALK", "TRANSIT"],
    "percentiles": [50],
    "grid_points_file": str(GRID_POINTS_FILE),
    "osm_file": str(OSM_FILE),
    "gtfs_file": str(GTFS_FILE),
    "chunk_count": int(len(manifest)),
}

with open(OUTPUT_DIR / "run_metadata_ptA_ptB_3000m.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("Saved manifest files to", OUTPUT_DIR)
manifest.head()

In [ ]:
def lookup_precomputed_travel_time(from_id: str, to_id: str, departure_hour: int) -> float | None:
    departure_label = time_label(departure_hour)
    from_id = str(from_id)
    to_id = str(to_id)

    index = pd.read_parquet(OUTPUT_DIR / "origin_chunk_index.parquet")
    match = index[
        (index["departure_label"] == departure_label)
        & (index["from_id"].astype(str) == from_id)
    ]

    if match.empty:
        raise ValueError(f"No precomputed chunk found for origin {from_id} at {departure_label}.")

    chunk = pd.read_parquet(match.iloc[0]["file"])
    row = chunk[
        (chunk["from_id"].astype(str) == from_id)
        & (chunk["to_id"].astype(str) == to_id)
    ]

    if row.empty or pd.isna(row.iloc[0]["travel_time"]):
        return None

    return float(row.iloc[0]["travel_time"])